# Where RL is used, and where it is not

You run the lesson's Taxi code yourself: drive Gymnasium's Taxi-v4 by hand, compare a random agent with a Q-learning agent, and then watch the same agent game a +5 per-pickup bonus. About 5 minutes.

Run the cells from top to bottom (Runtime > Run all works too). No account, key or GPU is needed.

## 1. Install Gymnasium

Gymnasium is the library that gives every RL practice world (an **environment**) the same interface. The lesson was run with version 1.4.0, which has `Taxi-v4`. Older versions only have `Taxi-v3`, so the next cell installs exactly 1.4.0.

Look for: no output, or a short pip message. If Colab asks you to restart the runtime, restart and run from here again.

In [1]:
%pip install -q gymnasium==1.4.0

Note: you may need to restart the kernel to use updated packages.


## 2. Start an episode

`gym.make("Taxi-v4")` builds the environment. `reset(seed=0)` starts an episode; the seed makes the start the same as in the lesson. The observation is one number that packs (taxi row, taxi column, passenger, destination) into one id.

Look for: observation `314`, decoded as `[3, 0, 3, 2]`: the taxi at row 3, column 0, the passenger at pad Y (3), the destination pad B (2). The time limit is `200` steps. In the map, the highlighted square is the taxi.

In [2]:
import gymnasium as gym
import numpy as np

print("gymnasium", gym.__version__, "| numpy", np.__version__)
ACTIONS = ["south", "north", "east", "west", "pickup", "dropoff"]
PADS = ["R", "G", "Y", "B"]  # passenger 4 means: in the taxi

env = gym.make("Taxi-v4", render_mode="ansi")
obs, info = env.reset(seed=0)
print("reset obs:", obs)
print("decoded (taxi_row, taxi_col, passenger, destination):", list(env.unwrapped.decode(obs)))
print("action_space:", env.action_space, "| observation_space:", env.observation_space)
print("max_episode_steps:", env.spec.max_episode_steps)
print(env.render())

gymnasium 1.4.0 | numpy 2.5.4
reset obs: 314
decoded (taxi_row, taxi_col, passenger, destination): [3, 0, 3, 2]
action_space: Discrete(6) | observation_space: Discrete(500)
max_episode_steps: 200
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




## 3. Four hand-picked steps

`step(action)` does one action and returns five things: the new observation, the reward, `terminated`, `truncated` and `info`. The rules: -1 per step, +20 for a delivery, -10 for a pickup or dropoff that is not allowed.

Look for: `pickup` gives `-10` (the passenger is not here), `west` gives `-1` and the observation stays `314` (a wall), `north` gives `-1` and the observation becomes `214`, `dropoff` gives `-10` (the taxi is empty).

In [3]:
for name, action in [("pickup", 4), ("west", 3), ("north", 1), ("dropoff", 5)]:
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"step({name:8s}) obs {obs}  decoded {list(env.unwrapped.decode(obs))}  reward {reward:4}  "
          f"terminated {terminated}  truncated {truncated}")

step(pickup  ) obs 314  decoded [3, 0, 3, 2]  reward  -10  terminated False  truncated False
step(west    ) obs 314  decoded [3, 0, 3, 2]  reward   -1  terminated False  truncated False
step(north   ) obs 214  decoded [2, 0, 3, 2]  reward   -1  terminated False  truncated False
step(dropoff ) obs 214  decoded [2, 0, 3, 2]  reward  -10  terminated False  truncated False


## 4. The baseline: a random agent

Before training anything, measure an agent that picks every action at random, on 100 episodes with start seeds 1000 to 1099. An episode ends when `terminated` (a delivery) or `truncated` (the 200-step limit) is True.

Look for: mean return `-765.1`, only `6` of 100 episodes delivered, mean steps `195.0` (most episodes run out of time).

In [4]:
env = gym.make("Taxi-v4")
env.action_space.seed(0)
returns, delivered, lengths = [], 0, []
for i in range(100):
    obs, info = env.reset(seed=1000 + i)
    total, steps, done = 0, 0, False
    while not done:
        obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
        total += reward
        steps += 1
        done = terminated or truncated
    returns.append(total)
    delivered += int(terminated)
    lengths.append(steps)
print("RANDOM agent, 100 episodes")
print("  mean return:", round(float(np.mean(returns)), 1))
print("  delivered:  ", delivered, "of 100")
print("  mean steps: ", round(float(np.mean(lengths)), 1))

RANDOM agent, 100 episodes
  mean return: -765.1
  delivered:   6 of 100
  mean steps:  195.0


## 5. The learner: tabular Q-learning

This is the training function from the lesson's runs. It keeps a table with one row per state (500) and one value per action (6). On each step it explores with probability `eps` (10% by default), otherwise takes the best-valued action, and nudges the table towards `reward + gamma * best value of the next state`.

It also counts what we care about besides the reward: deliveries and pickups. If the environment reports a `true_reward` in `info` (we use that later), it adds that up separately. Every `block` episodes it returns one row of averages.

Look for: no output. This cell only defines the function.

In [5]:
def train(env, episodes, alpha=0.1, gamma=0.99, eps=0.1, block=250, seed=0, q=None, start=0):
    """Q-learning. Returns the Q-table and one row per block of episodes:
    (last episode number, mean reward the agent was trained on, mean true reward, deliveries per episode, pickups per episode)."""
    rng = np.random.default_rng(seed)
    if q is None:
        q = np.zeros((env.observation_space.n, env.action_space.n))
    rows, acc = [], []
    for ep in range(start, start + episodes):
        obs, info = env.reset(seed=ep)
        total = true_total = 0.0
        delivered = pickups = 0
        done = False
        while not done:
            if rng.random() < eps:
                action = int(rng.integers(env.action_space.n))
            else:
                action = int(np.argmax(q[obs]))
            nxt, reward, terminated, truncated, info = env.step(action)
            target = reward + (0.0 if terminated else gamma * np.max(q[nxt]))
            q[obs, action] += alpha * (target - q[obs, action])
            total += reward
            true_total += info.get("true_reward", reward)
            delivered += int(terminated)
            pickups += int(action == 4 and list(env.unwrapped.decode(obs))[2] < 4 and list(env.unwrapped.decode(nxt))[2] == 4)
            obs = nxt
            done = terminated or truncated
        acc.append((total, true_total, delivered, pickups))
        if (ep + 1 - start) % block == 0:
            a = np.array(acc)
            rows.append((ep + 1, round(a[:, 0].mean(), 1), round(a[:, 1].mean(), 1), round(a[:, 2].mean(), 2), round(a[:, 3].mean(), 1)))
            acc = []
    return q, rows


def print_curve(rows, title):
    print(title)
    print(f"{'after':>6} {'trained reward':>15} {'true reward':>12} {'deliv.':>7} {'pick.':>6}")
    for ep, tr, true, deliv, pick in rows:
        print(f"{ep:>6} {tr:>15} {true:>12} {deliv:>7} {pick:>6}")

## 6. Train for 3,000 episodes and read the curve

Each row averages 250 training episodes, with 10% of moves still random. Here the trained reward and the true reward are the same thing (no bonus yet), so the two columns match.

Look for: the mean return climbs from `-250.8` to above zero (`2.4` in the last row). From 1,000 episodes on, deliveries per episode read `1.0`, yet the return keeps rising: the agent is finding shorter routes. This takes about 10 seconds.

In [6]:
env = gym.make("Taxi-v4")
q_table, rows = train(env, 3000, block=250)
print_curve(rows, "Q-learning on Taxi-v4, rows of 250 episodes")

Q-learning on Taxi-v4, rows of 250 episodes
 after  trained reward  true reward  deliv.  pick.
   250          -250.8       -250.8    0.42    1.4
   500          -118.4       -118.4    0.91    1.1
   750           -46.5        -46.5    0.99    1.0
  1000           -18.8        -18.8     1.0    1.0
  1250            -4.9         -4.9     1.0    1.0
  1500            -0.9         -0.9     1.0    1.0
  1750             1.0          1.0     1.0    1.0
  2000             1.8          1.8     1.0    1.0
  2250             2.6          2.6     1.0    1.0
  2500             1.7          1.7     1.0    1.0
  2750             2.0          2.0     1.0    1.0
  3000             2.4          2.4     1.0    1.0


## 7. Test it with exploring switched off

`run_episode` is the code from the lesson's code screen: at every step it takes the action with the highest value in the Q-table, until `terminated` or `truncated`. We run it on the same 100 start seeds the random agent had.

Look for: mean return `3.56`, mean steps `17.02` (the random agent needed `195.0`). But the worst return is `-200`: `2` episodes (seeds 1003 and 1060) never delivered and were cut off at 200 steps. A good mean can hide bad cases.

In [7]:
def run_episode(env, q_table, seed):
    obs, info = env.reset(seed=seed)
    total_reward, steps = 0, 0
    while True:
        action = int(np.argmax(q_table[obs]))
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        steps += 1
        if terminated or truncated:
            return total_reward, steps


results = [run_episode(env, q_table, 1000 + i) for i in range(100)]
print("GREEDY agent, 100 episodes")
print("  mean return:", round(float(np.mean([r for r, _ in results])), 2))
print("  mean steps: ", round(float(np.mean([s for _, s in results])), 2))
print("  worst return:", min(r for r, _ in results))
print("  hit the 200-step limit:", [(1000 + i, r, s) for i, (r, s) in enumerate(results) if s == 200], "(seed, return, steps)")

GREEDY agent, 100 episodes
  mean return: 3.56
  mean steps:  17.02
  worst return: -200
  hit the 200-step limit: [(1003, -200, 200), (1060, -200, 200)] (seed, return, steps)


## 8. Watch one episode, step by step

The same loop as `run_episode`, on episode seed 0, but printing every step. This is the episode you stepped through in the lesson.

Look for: the taxi drives to pad Y, picks up at step 7, drives to pad B and drops off at step 15 with reward `+20` and `terminated True`. The episode returns `(6, 15)`: 14 steps at -1, then +20.

In [8]:
obs, info = env.reset(seed=0)
print("start: obs", obs, "decoded", list(env.unwrapped.decode(obs)))
total_reward, steps = 0, 0
while True:
    action = int(np.argmax(q_table[obs]))
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    print(f"step {steps:2}: {ACTIONS[action]:8s} obs {obs:3}  decoded {list(env.unwrapped.decode(obs))}  "
          f"reward {reward:3}  total {total_reward:4}  terminated {terminated}  truncated {truncated}")
    if terminated or truncated:
        break
print("run_episode returns:", run_episode(env, q_table, 0))

start: obs 314 decoded [3, 0, 3, 2]
step  1: north    obs 214  decoded [2, 0, 3, 2]  reward  -1  total   -1  terminated False  truncated False
step  2: east     obs 234  decoded [2, 1, 3, 2]  reward  -1  total   -2  terminated False  truncated False
step  3: east     obs 254  decoded [2, 2, 3, 2]  reward  -1  total   -3  terminated False  truncated False
step  4: east     obs 274  decoded [2, 3, 3, 2]  reward  -1  total   -4  terminated False  truncated False
step  5: south    obs 374  decoded [3, 3, 3, 2]  reward  -1  total   -5  terminated False  truncated False
step  6: south    obs 474  decoded [4, 3, 3, 2]  reward  -1  total   -6  terminated False  truncated False
step  7: pickup   obs 478  decoded [4, 3, 4, 2]  reward  -1  total   -7  terminated False  truncated False
step  8: north    obs 378  decoded [3, 3, 4, 2]  reward  -1  total   -8  terminated False  truncated False
step  9: north    obs 278  decoded [2, 3, 4, 2]  reward  -1  total   -9  terminated False  truncated False
s

## 9. Broken on purpose: a bonus for every pickup

The startup's first idea was to pay couriers for every parcel picked up. This wrapper does the same to the taxi: +5 on every legal pickup, on top of Taxi's own reward. It keeps Taxi's own reward in `info["true_reward"]`, so we can still see the job we actually want.

Look for: no output. This cell only defines the wrapper.

In [9]:
class PickupBonus(gym.Wrapper):
    """The broken reward: +bonus on every legal pickup, like paying a courier per parcel collected.
    The environment's own reward is kept and also reported in info["true_reward"]."""

    def __init__(self, env, bonus=5.0):
        super().__init__(env)
        self.bonus = bonus

    def step(self, action):
        before = self.env.unwrapped.s
        obs, reward, terminated, truncated, info = self.env.step(action)
        _, _, pass_before, _ = self.env.unwrapped.decode(before)
        _, _, pass_after, _ = self.env.unwrapped.decode(obs)
        picked = action == 4 and pass_before < 4 and pass_after == 4
        info = dict(info, true_reward=reward, picked_up=picked)
        return obs, reward + (self.bonus if picked else 0.0), terminated, truncated, info

## 10. Train with the bonus

First 1,500 episodes on Taxi's own reward. Then the +5 pickup bonus is switched on, and random moves rise from 10% to 20%, for 8,500 more episodes. Each row averages 500 episodes. This takes about a minute.

Look for: until 5,000 episodes the trained reward sits about 5 above the true reward (one pickup per delivery). At `5500` pickups jump to `2.5` per episode while deliveries are still `1.0`: the agent has started gaming the bonus. By `10000` the trained reward is `100.2`, the true reward `-267.0`, and deliveries `0.0`.

In [10]:
q_bonus, phase1 = train(gym.make("Taxi-v4"), 1500, block=500)
q_phase1 = q_bonus.copy()  # kept for the experiments below
q_bonus, phase2 = train(PickupBonus(gym.make("Taxi-v4"), 5.0), 8500, block=500, q=q_bonus, start=1500, seed=1, eps=0.2)
print_curve(phase1 + phase2, "Bonus switched on after 1,500 episodes, rows of 500 episodes")

Bonus switched on after 1,500 episodes, rows of 500 episodes
 after  trained reward  true reward  deliv.  pick.
   500          -184.6       -184.6    0.67    1.2
  1000           -32.6        -32.6     1.0    1.0
  1500            -2.9         -2.9     1.0    1.0
  2000            -1.0         -6.1     1.0    1.0
  2500            -0.0         -5.2     1.0    1.0
  3000             0.9         -4.2     1.0    1.0
  3500             0.7         -4.6     1.0    1.1
  4000             0.8         -4.4     1.0    1.0
  4500             0.0         -5.1     1.0    1.0
  5000             0.1         -5.2     1.0    1.0
  5500             2.7         -9.8     1.0    2.5
  6000             9.2        -21.0    0.96    6.0
  6500            27.0        -66.7    0.82   18.8
  7000            35.9        -96.6    0.69   26.5
  7500            73.1       -192.6    0.34   53.1
  8000            95.1       -251.3    0.09   69.3
  8500            96.3       -263.7    0.03   72.0
  9000           100.

## 11. What the bonus-trained agent does

Let the final agent drive episode seed 0 with no random moves, and print what it does.

Look for: it picks the passenger up at pad Y, drives to pad G (row 0, column 4, not the destination B), drops the passenger there (a legal -1 step) and then picks up and drops on the spot, again and again. `95` pickups and `94` dropoffs, and the episode ends with `truncated True` at 200 steps. Each pickup pays -1 + 5 = `4.0` in trained reward, but the true reward stays `-1`.

In [11]:
benv = PickupBonus(gym.make("Taxi-v4"), 5.0)
obs, info = benv.reset(seed=0)
acts, done = [], False
while not done:
    a = int(np.argmax(q_bonus[obs]))
    obs, r, term, trunc, info = benv.step(a)
    acts.append((ACTIONS[a], list(benv.unwrapped.decode(obs)), r, info["true_reward"]))
    done = term or trunc
print("bonus agent, seed 0: steps", len(acts), "| terminated", term, "| truncated", trunc)
print("first 18 steps (action, decoded obs, trained reward, true reward):")
for x in acts[:18]:
    print("  ", x)
print("pickups", sum(1 for x in acts if x[0] == "pickup"), "| dropoffs", sum(1 for x in acts if x[0] == "dropoff"))
print("trained reward", sum(x[2] for x in acts), "| true reward", sum(x[3] for x in acts))

bonus agent, seed 0: steps 200 | terminated False | truncated True
first 18 steps (action, decoded obs, trained reward, true reward):
   ('north', [2, 0, 3, 2], -1.0, -1)
   ('east', [2, 1, 3, 2], -1.0, -1)
   ('east', [2, 2, 3, 2], -1.0, -1)
   ('east', [2, 3, 3, 2], -1.0, -1)
   ('south', [3, 3, 3, 2], -1.0, -1)
   ('south', [4, 3, 3, 2], -1.0, -1)
   ('pickup', [4, 3, 4, 2], 4.0, -1)
   ('north', [3, 3, 4, 2], -1.0, -1)
   ('north', [2, 3, 4, 2], -1.0, -1)
   ('north', [1, 3, 4, 2], -1.0, -1)
   ('north', [0, 3, 4, 2], -1.0, -1)
   ('east', [0, 4, 4, 2], -1.0, -1)
   ('dropoff', [0, 4, 1, 2], -1.0, -1)
   ('pickup', [0, 4, 4, 2], 4.0, -1)
   ('dropoff', [0, 4, 1, 2], -1.0, -1)
   ('pickup', [0, 4, 4, 2], 4.0, -1)
   ('dropoff', [0, 4, 1, 2], -1.0, -1)
   ('pickup', [0, 4, 4, 2], 4.0, -1)
pickups 95 | dropoffs 94
trained reward 275.0 | true reward -200


## Now change it

Three small experiments. Each cell is ready to run. Edit the line marked `# <- edit`, run the cell again, and compare.

### Experiment 1: keep random moves at 10%

The marked line sets random moves in the bonus phase to 10% (the lesson's run used 20%). With fewer random moves the agent tries the pickup-drop loop less often.

What should change: after 10,000 episodes this agent has not found the trick yet: about `1.0` pickup and `1.0` delivery per episode, last row `[10000, 7.9, 2.8, 1.0, 1.0]`. The reward still pays more for the loop, so the bug is only hidden, not fixed. Set it back to `0.2` to see the trick return. About a minute.

In [12]:
EPS_BONUS = 0.1  # <- edit: the lesson's run used 0.2
q, rows = train(PickupBonus(gym.make("Taxi-v4"), 5.0), 8500, block=500, q=q_phase1.copy(), start=1500, seed=1, eps=EPS_BONUS)
print_curve(rows, f"Bonus phase with {EPS_BONUS:.0%} random moves")

Bonus phase with 10% random moves
 after  trained reward  true reward  deliv.  pick.
  2000             6.8          1.6     1.0    1.0
  2500             7.3          2.2     1.0    1.0
  3000             6.9          1.9     1.0    1.0
  3500             7.5          2.4     1.0    1.0
  4000             7.9          2.8     1.0    1.0
  4500             7.2          2.2     1.0    1.0
  5000             7.1          2.0     1.0    1.0
  5500             7.7          2.7     1.0    1.0
  6000             7.8          2.7     1.0    1.0
  6500             7.6          2.5     1.0    1.0
  7000             7.4          2.3     1.0    1.0
  7500             7.4          2.3     1.0    1.0
  8000             7.4          2.3     1.0    1.0
  8500             7.3          2.2     1.0    1.0
  9000             7.8          2.7     1.0    1.0
  9500             7.7          2.6     1.0    1.0
 10000             7.9          2.8     1.0    1.0


### Experiment 2: pay for the outcome

This wrapper puts the same +5 on deliveries instead of pickups. The marked line picks which wrapper the bonus phase trains with (20% random moves, as in the lesson).

What should change: with `DeliveryBonus` the trained reward is the true reward plus 5 in every row, pickups stay at about `1.0` and deliveries at `1.0`, because the bonus cannot be collected without doing the job. Change the line to `PickupBonus` and the gaming from step 10 comes back.

In [13]:
class DeliveryBonus(gym.Wrapper):
    """The fixed reward: +bonus only when the passenger is delivered (terminated is True)."""

    def __init__(self, env, bonus=5.0):
        super().__init__(env)
        self.bonus = bonus

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        info = dict(info, true_reward=reward)
        return obs, reward + (self.bonus if terminated else 0.0), terminated, truncated, info


WRAPPER = DeliveryBonus  # <- edit: try PickupBonus
q, rows = train(WRAPPER(gym.make("Taxi-v4"), 5.0), 8500, block=500, q=q_phase1.copy(), start=1500, seed=1, eps=0.2)
print_curve(rows, f"Bonus phase with {WRAPPER.__name__}")

Bonus phase with DeliveryBonus
 after  trained reward  true reward  deliv.  pick.
  2000            -1.1         -6.1     1.0    1.0
  2500            -0.2         -5.2     1.0    1.0
  3000             0.8         -4.2     1.0    1.0
  3500             0.4         -4.6     1.0    1.1
  4000             0.6         -4.4     1.0    1.0
  4500            -0.1         -5.1     1.0    1.0
  5000            -0.2         -5.2     1.0    1.0
  5500            -0.5         -5.5     1.0    1.0
  6000             0.8         -4.2     1.0    1.1
  6500             0.3         -4.7     1.0    1.0
  7000             0.3         -4.7     1.0    1.0
  7500            -0.2         -5.2     1.0    1.0
  8000             0.3         -4.7     1.0    1.0
  8500            -0.2         -5.2     1.0    1.0
  9000            -0.0         -5.0     1.0    1.0
  9500            -0.7         -5.7     1.0    1.0
 10000             0.6         -4.4     1.0    1.0


### Experiment 3: train for less

The marked line sets how many episodes the plain agent trains before the greedy test of step 7 (the lesson used 3,000).

What should change: after only 500 episodes the agent has not learned the whole map yet. Our run: `91` of 100 test episodes hit the 200-step limit and the mean return fell to `-234.78` (against `2` and `3.56` after 3,000). Try 1000 or 2000 and watch both numbers improve. RL needs many trials even on a 500-state toy, which is why the first question is where the agent practises.

In [14]:
EPISODES = 500  # <- edit: the lesson used 3000
q_short, _ = train(gym.make("Taxi-v4"), EPISODES, block=250)
res = [run_episode(env, q_short, 1000 + i) for i in range(100)]
print(f"GREEDY agent after {EPISODES} training episodes, 100 test episodes")
print("  mean return:", round(float(np.mean([r for r, _ in res])), 2))
print("  mean steps: ", round(float(np.mean([s for _, s in res])), 2))
print("  hit the 200-step limit:", sum(s == 200 for _, s in res), "of 100")

GREEDY agent after 500 training episodes, 100 test episodes
  mean return: -234.78
  mean steps:  182.76
  hit the 200-step limit: 91 of 100


## What you should see

- `env.reset(seed=0)` on Taxi-v4 returns observation 314, and a pickup right away returns reward -10.
- A random agent over 100 episodes has a mean return far below zero (-765.1) and delivers only rarely (6 of 100).
- The Q-learning curve climbs from about -251 to above zero, and deliveries per episode reach 1.0 within the first 1,000 or so episodes.
- The greedy test averages under 20 steps per episode (17.02); a few episodes (2 of 100) still hit the 200-step limit.
- With the +5 pickup bonus, the trained reward ends far above the true reward (100.2 against -267.0) and deliveries per episode fall towards 0.